# 02 — Amenity Composition (Paris)

Queries OSM for amenities and computes density and food/drink ratio per grid cell.

**Data source:** Overpass API (OSM) — fully portable, works for any city.

**Method:** Single batch Overpass query for all amenity/shop/office/craft/tourism/leisure POIs
in the Paris bounding box, then BallTree matching to assign each POI to its nearest grid cell.

**Output columns:** `cell_id`, `amenity_density`, `amenity_ratio_food_drink`

**Output file:** `csv/Paris/02_amenity_composition.csv`

In [1]:
PARIS_CONFIG = "paris.json"

In [2]:
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("cache", exist_ok=True)

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M   = config["grid_cell_size_m"]
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2
QUERY_RADIUS  = 500
CSV_DIR       = config["csv_dir"]
os.makedirs(CSV_DIR, exist_ok=True)

df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
print(f"Loaded {len(df_grid)} grid cells")

BUFFER  = 0.015
LAT_MIN = df_grid["cell_lat"].min() - BUFFER
LAT_MAX = df_grid["cell_lat"].max() + BUFFER
LON_MIN = df_grid["cell_lon"].min() - BUFFER
LON_MAX = df_grid["cell_lon"].max() + BUFFER
BBOX    = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"
print(f"Bounding box: {BBOX}")

Loaded 120331 grid cells
Bounding box: 48.1103076,1.4372832000000002,49.2511184,3.5356377


In [3]:
# ── Overpass helper (cached) ──────────────────────────
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "paris-grid/1.0 (research project)"}

def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"

def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            import time; time.sleep(5 + attempt * 3)
    raise RuntimeError(f"Overpass failed: {last_error}")

print("Overpass helper ready.")

Overpass helper ready.


In [4]:
# ── Amenity tags and food/drink categories ────────────
QUERY_TAGS = ["amenity", "shop", "office", "craft", "tourism", "leisure"]

FOOD_DRINK_VALUES = {
    "restaurant", "fast_food", "cafe", "bar", "pub", "food_court", "bakery",
    "ice_cream", "deli", "confectionery", "coffee", "tea", "beverages",
    "butcher", "greengrocer", "pastry", "brewery", "grocery", "seafood",
    # French-specific additions
    "boulangerie", "patisserie", "brasserie", "bistro",
}

In [5]:
# ── Batch query: ALL amenity POIs in Paris bbox ───────
tag_filters = "\n ".join(
    [f'node["{tag}"]({BBOX});' for tag in QUERY_TAGS]
    + [f'way["{tag}"]({BBOX});' for tag in QUERY_TAGS]
)
query = f"[out:json][timeout:120];\n(\n {tag_filters}\n);\nout center tags;"

print("Querying all amenity POIs in Paris...")
data = query_overpass_cached(query)

EARTH_RADIUS_M = 6371000
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
    lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
    if not (lat and lon):
        continue
    has_tag = False
    is_food = False
    for tag_key in QUERY_TAGS:
        if tag_key in tags:
            has_tag = True
            if tags[tag_key] in FOOD_DRINK_VALUES:
                is_food = True
            break
    if has_tag:
        poi_records.append({"lat": float(lat), "lon": float(lon), "is_food": is_food})

print(f"Found {len(poi_records)} POIs ({sum(p['is_food'] for p in poi_records)} food/drink)")

Querying all amenity POIs in Paris...


Found 547916 POIs (42055 food/drink)


In [6]:
# ── BallTree: assign each POI to nearest grid cell ────
cell_coords_rad = np.radians(df_grid[["cell_lat", "cell_lon"]].values)
cell_ids  = df_grid["cell_id"].tolist()
cell_total = {c: 0 for c in cell_ids}
cell_food  = {c: 0 for c in cell_ids}

if poi_records:
    cell_tree = BallTree(cell_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = cell_tree.query(poi_coords, k=1)

    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            cid = cell_ids[idx]
            cell_total[cid] += 1
            if poi_records[j]["is_food"]:
                cell_food[cid] += 1

records = []
for cid in cell_ids:
    total = cell_total[cid]
    records.append({
        "cell_id": cid,
        "amenity_density": round(total / CELL_AREA_KM2, 2),
        "amenity_ratio_food_drink": round(cell_food[cid] / total, 4) if total > 0 else 0.0,
    })

df_amenities = pd.DataFrame(records)
print(f"Completed: {len(df_amenities)} cells")
print(f"Mean amenity density: {df_amenities['amenity_density'].mean():.1f} per km2")
print(f"Cells with zero amenities: {(df_amenities['amenity_density'] == 0).sum()}")

Completed: 120331 cells
Mean amenity density: 189.6 per km2
Cells with zero amenities: 55860


In [7]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/02_amenity_composition.csv"
df_amenities.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_amenities)} rows x {df_amenities.shape[1]} cols)")
print(df_amenities.describe().round(2).to_string())

Saved: csv/Paris/02_amenity_composition.csv  (120331 rows x 3 cols)
       amenity_density  amenity_ratio_food_drink
count        120331.00                 120331.00
mean            189.63                      0.02
std             696.51                      0.10
min               0.00                      0.00
25%               0.00                      0.00
50%              44.44                      0.00
75%             133.33                      0.00
max          107333.33                      1.00
